In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install faiss-cpu sentence-transformers langchain langchain-community

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-vlqif8vx/unsloth_9098ec9b6ad14412805ed355d0447451
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-vlqif8vx/unsloth_9098ec9b6ad14412805ed355d0447451
  Resolved https://github.com/unslothai/unsloth.git to commit 2f1cc712e9c1fd02601e5a686e3701c23b0d8577
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 124.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 103.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 122.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 57.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 119.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.9 MB/s  0:00:00
  Created wheel for u

In [4]:
import json
import torch
import os
import re
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


print("Λήψη FAISS Database από Hugging Face...")
hf_repo = "deanwashburn/Qwen2.5-7B-ERP-Order-Extraction"

os.makedirs("erp_faiss_index 2", exist_ok=True)
hf_hub_download(repo_id=hf_repo, filename="erp_faiss_index 2/index.faiss", local_dir="./")
hf_hub_download(repo_id=hf_repo, filename="erp_faiss_index 2/index.pkl", local_dir="./")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vector_db = FAISS.load_local(
    folder_path="./erp_faiss_index 2",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

print("Φόρτωση Qwen 7B Model...")
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deanwashburn/Qwen2.5-7B-ERP-Order-Extraction",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def normalize_gr(text):
    trans = str.maketrans("άέήίόύώϊϋΐΰ", "αεηιουωιυιυ")
    text = text.lower().translate(trans)

    text = text.replace("ντοματ", "τοματ")

    words = re.findall(r'[α-ωa-z]{3,}|[0-9]{2,}', text)
    stop = {"απο", "τον", "την", "στον", "στην", "στη", "για", "και", "θελω", "θελουμε", "στειλτε", "στειλε", "βαλε", "παραγγελια", "ειμαι", "χρειαζομαι", "επειγοντως", "κιλα", "σακια", "σακι", "σακος", "τσουβαλια", "τσουβαλι", "παλετες", "τονους", "μπουκαλια", "μπουκαλι", "σακουλακια", "σακουλακι", "ονομαζομαι", "γεια", "σας", "μου", "συντομοτερο", "παρακαλω", "ζητησε", "σημειωσε", "πληρωσει", "μετρητα", "προσεξε", "ειναι", "καλα", "κλεισμενα", "γιατι", "βρεχει", "100", "500", "150", "1000"}
    return [w[:5] for w in words if w not in stop]

all_docs = list(vector_db.docstore._dict.values())


def process_order_to_json(raw_order_text):

    sub_queries = [q.strip() for q in re.split(r'[,.:;]|\bκαι\b', raw_order_text) if len(q.strip()) > 3]
    sub_queries.append(raw_order_text)

    order_stems_list = normalize_gr(raw_order_text)
    order_stems = set(order_stems_list)


    best_client = None
    best_client_overlap = 0

    for doc in all_docs:
        meta = doc.metadata
        if meta.get("type") == "client":
            cname = meta.get("name", "")
            clean_cname = re.sub(r'\(Κατ\.\s*\d+\)', '', cname).strip()
            c_stems_list = normalize_gr(clean_cname)
            c_stems = set(c_stems_list)
            overlap = len(order_stems.intersection(c_stems))

            if len(c_stems) == 1 and len(c_stems_list) >= 2:
                stem = c_stems_list[0]
                if order_stems_list.count(stem) >= 2:
                    overlap = 2
            if overlap >= 2 and overlap > best_client_overlap:
                best_client_overlap = overlap
                best_client = meta

    if best_client:
        client_info = f"ΠΕΛΑΤΗΣ: {best_client.get('name')} | ΑΦΜ: {best_client.get('afm')}"
    else:
        client_info = "ΔΕΝ ΒΡΕΘΗΚΕ ΠΕΛΑΤΗΣ ΣΤΗΝ ΑΠΟΘΗΚΗ. Βάλε afm: 'N/A'"


    candidate_products = {}
    for sq in sub_queries:
        results = vector_db.similarity_search_with_score(sq, k=4)
        for doc, score in results:
            meta = doc.metadata
            if meta.get("type") == "product" and score < 11.5:
                sku = meta.get("sku")
                if sku not in candidate_products or score < candidate_products[sku][1]:
                    candidate_products[sku] = (meta, score)

    scored_products = []
    for doc in all_docs:
        meta = doc.metadata
        if meta.get("type") == "product":
            pname = meta.get("product_name", "")
            clean_pname = re.sub(r'\(Κωδ\.\s*\d+\)', '', pname)
            p_stems = set(normalize_gr(clean_pname))
            overlap = len(order_stems.intersection(p_stems))
            if overlap >= 2:
                sku = meta.get("sku")
                vec_score = candidate_products[sku][1] if sku in candidate_products else 10.0
                is_golden = 1 if "Κωδ." not in pname else 0
                scored_products.append((overlap, is_golden, -vec_score, meta))

    scored_products.sort(reverse=True, key=lambda x: (x[0], x[1], x[2]))
    top_products = [item[3] for item in scored_products[:5]]

    if top_products:
        valid_products = [
            f"ΠΡΟΪΟΝ: {m.get('product_name')} | SKU: {m.get('sku')} | BARCODE: {m.get('barcode')} | ΤΙΜΗ: {m.get('unit_price')}"
            for m in top_products
        ]
        products_info = "\n".join(valid_products)
        print(f"[RAG Debug] Βρέθηκαν {len(valid_products)} ακριβείς ταυτίσεις προϊόντων στη βάση.")
    else:
        print("[RAG Security] Αδύναμη ταύτιση προϊόντων. Κλείδωμα κωδικών σε N/A.")
        products_info = "ΚΑΝΕΝΑ ΠΡΟΪΟΝ ΔΕΝ ΒΡΕΘΗΚΕ. Βάλε sku: 'N/A', barcode: 'N/A' και unit_price: 0.0"

    input_text = f"Δεδομένα Πελάτη:\n{client_info}\n\nΔεδομένα Αποθήκης:\n{products_info}\n\nΠαραγγελία: {raw_order_text}"

    instruction = """Εξάγαγε τα δεδομένα της παραγγελίας συμπληρώνοντας ΑΥΣΤΗΡΑ το JSON template.
ΚΑΝΟΝΕΣ:
1. Στο "name" βάλε το Ονοματεπώνυμο ή την Επωνυμία του πελάτη.
2. Στο "afm" βάλε το ΑΦΜ από τα 'Δεδομένα Πελάτη'. Αν δεν υπάρχει, βάλε 'N/A'.
3. Για κάθε αντικείμενο της παραγγελίας, βρες το αντίστοιχο "ΠΡΟΪΟΝ" στα 'Δεδομένα Αποθήκης' και αντέγραψε ΑΚΡΙΒΩΣ το SKU, το BARCODE και την ΤΙΜΗ (unit_price).
4. Κράτησε στο "quantity" ΑΚΡΙΒΩΣ τον αριθμό που ζητάει ο πελάτης στην παραγγελία (μην τον διαιρείς με τα κιλά της συσκευασίας).
5. Αν ένα προϊόν της παραγγελίας ΔΕΝ υπάρχει στα 'Δεδομένα Αποθήκης', βάλε sku: "N/A", barcode: "N/A" και unit_price: 0.0.

Αναμενόμενο JSON Template:
{
  "customer": {
    "name": "<ΟΝΟΜΑ ΠΕΛΑΤΗ>",
    "afm": "<ΑΦΜ Ή 'N/A'>"
  },
  "order_items": [
    {
      "category": "<λιπάσματα, σπόροι, ή ζωοτροφές>",
      "item_description": "<ΠΕΡΙΓΡΑΦΗ>",
      "sku": "<SKU ΑΠΟ ΑΠΟΘΗΚΗ Ή 'N/A'>",
      "barcode": "<BARCODE ΑΠΟ ΑΠΟΘΗΚΗ Ή 'N/A'>",
      "quantity": <ΑΡΙΘΜΟΣ>,
      "unit": "<ΜΟΝΑΔΑ>",
      "unit_price": <ΤΙΜΗ ΜΟΝΑΔΑΣ (ΑΡΙΘΜΟΣ) Ή 0.0>,
      "total_price": 0.0
    }
  ],
  "total_order_value": 0.0
}"""


    inputs = tokenizer(
        [alpaca_prompt.format(instruction, input_text, "")],
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=0.0, do_sample=False)
    full_response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]


    raw_json_str = full_response.split("### Response:\n")[-1].strip()

    if raw_json_str.startswith("```json"):
        raw_json_str = raw_json_str[7:]
    elif raw_json_str.startswith("```"):
        raw_json_str = raw_json_str[3:]
    if raw_json_str.endswith("```"):
        raw_json_str = raw_json_str[:-3]

    try:
        parsed_json = json.loads(raw_json_str.strip())

        if "customer" in parsed_json and isinstance(parsed_json["customer"], dict):
            parsed_json["customer"]["afm"] = best_client.get("afm", "N/A") if best_client else "N/A"

        valid_sku_map = {m.get("sku"): m for m in top_products}

        total_value = 0.0
        for item in parsed_json.get("order_items", []):
            sku = item.get("sku", "N/A")
            if sku in valid_sku_map:
                matched_prod = valid_sku_map[sku]
                item["barcode"] = matched_prod.get("barcode", "N/A")
                item["unit_price"] = float(matched_prod.get("unit_price", 0.0))
            else:
                item["sku"] = "N/A"
                item["barcode"] = "N/A"
                item["unit_price"] = 0.0

            try:
                q = float(item.get("quantity", 0))
                up = float(item.get("unit_price", 0))
                item_total = round(q * up, 2)
                item["total_price"] = item_total
                total_value += item_total
            except (ValueError, TypeError):
                item["total_price"] = 0.0

        parsed_json["total_order_value"] = round(total_value, 2)
        parsed_json["delivery_notes"] = raw_order_text
        final_json = json.dumps(parsed_json, indent=2, ensure_ascii=False)
    except json.JSONDecodeError:
        final_json = raw_json_str.strip() + f'\n\n[Python Injection] "delivery_notes": "{raw_order_text}"'

    return final_json


demo_orders = [
    # --- 5 Παραγγελίες με Υπαρκτούς Πελάτες (Βάση RAG) ---
    "Από τον Αγροτικό Συνεταιρισμό Θεσσαλίας, θέλουμε 100 τσουβάλια νιτρική αμμωνία και 50 κιλά σπόρο για καλαμπόκι. Στείλτε τα Λάρισα.",
    "Είμαι ο Γεώργιος Παπαδόπουλος από Καρδίτσα. Χρειάζομαι επειγόντως 2 παλέτες φύραμα για κότες και 10 σακιά λίπασμα 20-20-20.",
    "Παραγγελία για Συνεταιρισμός Αμπελουργών Νάουσας: 500 κιλά σύνθετο λίπασμα NPK και 5 μπουκάλια υγρό λίπασμα.",
    "Βάλε για Δημήτριος Δημητρίου στη Λαμία, 20 κιλά θειική αμμωνία και 10 σακουλάκια σπόροι ντομάτας.",
    "Από την Αγροτική Εταιρεία 'Η Δήμητρα' Ο.Ε., στείλτε 30 σακιά φύραμα για χοίρους και 5 τόνους σπόρο ηλίανθου.",

    # --- 5 Παραγγελίες με Άγνωστους Πελάτες (Threshold Fallback -> 'N/A') ---
    "Γεια σας, ονομάζομαι Κώστας Λάμπρου. Θέλω να μου στείλετε 50 κιλά σπόρο σιτάρι. Το συντομότερο παρακαλώ.",
    "Η Μαρία Παπαδοπούλου ζήτησε 3 τσουβάλια τροφή για κουνέλια.",
    "Σημείωσε για την Ελένη Καρρά: 100 κιλά κριθάρι για τα ζώα. Θα τα πληρώσει μετρητά.",
    "Ο Γιώργος Παππάς θέλει 5 παλέτες μικτό φύραμα. Πρόσεξε να είναι καλά κλεισμένα γιατί βρέχει.",
    "Μήτσος Καραχάλιος: Θέλω 150 κιλά καλαμπόκι σπαστό (για τις κότες)."
]

print("\n" + "="*50)
print("=== ΕΝΑΡΞΗ ΑΥΤΟΜΑΤΗΣ ΕΠΙΔΕΙΞΗΣ (10 ΣΕΝΑΡΙΑ) ===")
print("="*50 + "\n")

for i, order in enumerate(demo_orders, 1):
    print(f"[{i}/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:\n{order}")
    print("-" * 50)

    result = process_order_to_json(order)

    print("ΠΑΡΑΓΟΜΕΝΟ JSON:")
    print(result)
    print("="*50 + "\n")

Λήψη FAISS Database από Hugging Face...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Φόρτωση Qwen 7B Model...
==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).

=== ΕΝΑΡΞΗ ΑΥΤΟΜΑΤΗΣ ΕΠΙΔΕΙΞΗΣ (10 ΣΕΝΑΡΙΑ) ===

[1/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Από τον Αγροτικό Συνεταιρισμό Θεσσαλίας, θέλουμε 100 τσουβάλια νιτρική αμμωνία και 50 κιλά σπόρο για καλαμπόκι. Στείλτε τα Λάρισα.
--------------------------------------------------
[RAG Debug] Βρέθηκαν 5 ακριβείς ταυτίσεις προϊόντων στη βάση.


... (14 outputs left out)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Ελένη Καρρά",
    "afm": "N/A"
  },
  "order_items": [
    {
      "category": "ζωοτροφές",
      "item_description": "Κριθάρι Κτηνοτροφικό για Ζώα (Σακί 40kg)",
      "sku": "FEED-003",
      "barcode": "5201000000011",
      "quantity": 100,
      "unit": "κιλά",
      "unit_price": 16.5,
      "total_price": 1650.0
    }
  ],
  "total_order_value": 1650.0,
  "delivery_notes": "Σημείωσε για την Ελένη Καρρά: 100 κιλά κριθάρι για τα ζώα. Θα τα πληρώσει μετρητά."
}

[9/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Ο Γιώργος Παππάς θέλει 5 παλέτες μικτό φύραμα. Πρόσεξε να είναι καλά κλεισμένα γιατί βρέχει.
--------------------------------------------------
[RAG Security] Αδύναμη ταύτιση προϊόντων. Κλείδωμα κωδικών σε N/A.


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Γιώργος Παππάς",
    "afm": "N/A"
  },
  "order_items": [
    {
      "category": "ζωοτροφές",
      "item_description": "Μικτό φύραμα",
      "sku": "N/A",
      "barcode": "N/A",
      "quantity": 5,
      "unit": "παλέτες",
      "unit_price": 0.0,
      "total_price": 0.0
    }
  ],
  "total_order_value": 0.0,
  "delivery_notes": "Ο Γιώργος Παππάς θέλει 5 παλέτες μικτό φύραμα. Πρόσεξε να είναι καλά κλεισμένα γιατί βρέχει."
}

[10/10] ΕΙΣΕΡΧΟΜΕΝΗ ΠΑΡΑΓΓΕΛΙΑ:
Μήτσος Καραχάλιος: Θέλω 150 κιλά καλαμπόκι σπαστό (για τις κότες).
--------------------------------------------------
[RAG Debug] Βρέθηκαν 1 ακριβείς ταυτίσεις προϊόντων στη βάση.
ΠΑΡΑΓΟΜΕΝΟ JSON:
{
  "customer": {
    "name": "Μήτσος Καραχάλιος",
    "afm": "N/A"
  },
  "order_items": [
    {
      "category": "ζωοτροφές",
      "item_description": "Καλαμπόκι σπαστό για κότες",
      "sku": "FEED-004",
      "barcode": "5201000000012",
      "quantity": 150,
      "unit": "κιλά",
